# فكرة مكتوبة → مشاهد واقعية مع صوت عربي أو إنجليزي
مسار تجريبي: **خمسة أسئلة → Qwen يخطط اللقطات → Qwen يكتب السكربت → XTTS يولّد الصوت → LTX يولّد المشاهد → MP4**.

اختر **Runtime → Change runtime type → T4 GPU** قبل البدء. إعداد البداية 288×512 و24 fps لتقليل الذاكرة؛ جودة أعلى تحتاج موارد أكثر. تنزيلات النماذج كبيرة (احتفظ بنحو 25 GB مساحة خالية). نجاح هذا المسار على Colab المجاني وجودة الفيديو الكامل **لم يُتحقق منهما بعد**. هنا اختُبر التخطيط على CPU والتجميع واستخدام API الفيديو بأوزان صغيرة عشوائية فقط.

لا توجد أوزان درّبناها من الصفر: نستخدم نماذج المشروع الأساسية Qwen وXTTS، ونضيف LTX-Video-0.9.5 للمشاهد. XTTS يخضع لترخيص CPML غير التجاري. اقرأ [الدليل](https://github.com/garaahmad/Voice2Reel/blob/main/docs/REALISTIC_VIDEO.md).

In [ ]:
from pathlib import Path
import zipfile
import os
import subprocess
import sys
from google.colab import files, drive

# Download Code -> Download ZIP from your private GitHub repository, then upload it here.
if not Path('/content/Voice2Reel/pyproject.toml').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one repository ZIP from GitHub')
    destination = Path('/content/repo_upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(info.file_size for info in archive.infolist()) > 50_000_000:
            raise ValueError('Expected source-only repository ZIP, under 50 MB')
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    projects = list(destination.rglob('pyproject.toml'))
    if len(projects) != 1:
        raise ValueError('Cannot identify repository root')
    import shutil
    shutil.copytree(projects[0].parent, '/content/Voice2Reel')
    del uploaded
os.chdir('/content/Voice2Reel')
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/Voice2Reel')
WORK.mkdir(parents=True, exist_ok=True)
print('Persistent artifacts:', WORK)


## تثبيت المتطلبات والتحقق من GPU

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'fonts-dejavu-core'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.8'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/content/v2r-env'], check=True)
PY = '/content/v2r-env/bin/python'
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args], check=True)
def run(*args):
    subprocess.run([PY, *map(str, args)], check=True)
pip_install('torch==2.8.0', 'torchaudio==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126')
pip_install('-r', 'requirements/voice.txt', '-r', 'requirements/video.txt')
run('-c', "import torch; print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slower)')")
# Isolated Python environment: no kernel restart or changes to Colab's preinstalled packages.

freeze = subprocess.check_output([sys.executable, "-m", "uv", "pip", "freeze", "--python", PY], text=True)
(WORK / "03_realistic_video_environment.txt").write_text(freeze)


## الفكرة والفيديو المرجعي الاختياري
المرجع يزوّد صورة بداية للمشهد الأول؛ لا يجري تحليل محتواه أو تقليد المونتاج تلقائيًا.

In [ ]:
#@title اكتب الفكرة
IDEA = "فيديو واقعي عن خطأ وضع فاصلة منقوطة بعد if في JavaScript" #@param {type:"string"}
USE_REFERENCE_VIDEO = False #@param {type:"boolean"}
REFERENCE_VIDEO = None
if USE_REFERENCE_VIDEO:
    reference_upload = files.upload()
    if len(reference_upload) != 1:
        raise ValueError('Upload one reference video')
    name = next(iter(reference_upload))
    saved = WORK / ('reference' + Path(name).suffix)
    saved.write_bytes(reference_upload[name])
    REFERENCE_VIDEO = str(saved)
    del reference_upload


## خمسة أسئلة فقط

In [ ]:
#@title إجاباتك
AUDIENCE = "مبتدئون في البرمجة" #@param {type:"string"}
DURATION = 15 #@param {type:"slider", min:10, max:40, step:1}
LANGUAGE = "ar" #@param ["ar", "en"]
SETTING = "A young adult programmer in a navy hoodie at a wooden desk, warm lamp, realistic home office." #@param {type:"string"}
MESSAGE = "الفاصلة المنقوطة مباشرة بعد if تنهي عبارتها وتجعل الكتلة التالية مستقلة. الفاصلتان في رأس for التقليدية صحيحتان. اشرح خطأ if ونصيحة مراجعة الشرط دون تعميمات." #@param {type:"string"}
import json
PROJECT = WORK / 'realistic'
PROJECT.mkdir(exist_ok=True)
brief_path = PROJECT / 'brief.input.json'
brief_path.write_text(json.dumps({
    'idea': IDEA, 'audience': AUDIENCE, 'target_duration_seconds': DURATION,
    'language': LANGUAGE, 'setting': SETTING, 'message': MESSAGE,
    'reference_video': REFERENCE_VIDEO,
}, ensure_ascii=False), encoding='utf-8')
run('-m', 'voice2reel.director', '--brief', brief_path, '--output', PROJECT, '--device', 'cuda')


## مراجعة الخطة والسكربت
راجع دقة المعلومات الآن. المخطط يمنع أخطاء البنية، لكنه لا يضمن صحة النص أو جمال المشاهد. يمكنك تعديل `script.json` قبل توليد الصوت؛ حافظ على عدد المقاطع وترتيبها.

In [ ]:
print((PROJECT / 'plan.json').read_text())
print((PROJECT / 'script.json').read_text())

## التعليق الصوتي الجاهز
ينتهي تشغيل كل نموذج قبل تحميل النموذج التالي لتوفير ذاكرة GPU.

In [ ]:
SPEAKER = "Claribel Dervla" #@param ["Claribel Dervla", "Andrew Chipper", "Ana Florence", "Daisy Studious"]
run('-m', 'voice2reel.voice.synthesize', '--script', PROJECT / 'script.json', '--speaker', SPEAKER,
    '--device', 'cuda', '--output', PROJECT / 'narration.wav')
from IPython.display import Audio, Video, display
display(Audio(str(PROJECT / 'narration.wav')))


## اختبر أول لقطة قبل توليد الفيديو كله
التنزيل الأول كبير. إذا انتهت ذاكرة T4 أو وقت الجلسة، احتفظ بملفات Drive؛ المقاطع المكتملة تُستأنف تلقائيًا عند ثبات الإعدادات. لا تعني كلمة واقعي أن هوية الشخص ستظل متطابقة بين كل اللقطات.

In [ ]:
WIDTH = 288 #@param {type:"integer"}
HEIGHT = 512 #@param {type:"integer"}
STEPS = 40 #@param {type:"slider", min:20, max:60, step:10}
video_args = ['-m', 'voice2reel.realistic', '--project', PROJECT,
              '--width', WIDTH, '--height', HEIGHT, '--steps', STEPS]
run(*video_args, '--max-clips', 1)
display(Video(str(PROJECT / 'clips' / '000.mp4'), embed=True))


## توليد بقية المشاهد وتجميع الفيديو
يُضاف الصوت والعناوين العربية برمجيًا. نموذج الفيديو لا يرسم كودًا مقروءًا داخل الشاشة. لا توجد مزامنة شفاه أو تحليل كامل للفيديو المرجعي.

In [ ]:
run(*video_args)
display(Video(str(PROJECT / 'video.mp4'), embed=True))
files.download(str(PROJECT / 'video.mp4'))
print('كل النتائج محفوظة في:', PROJECT)